# Notebook 01: Data Loading and Verification

This notebook loads the raw Freddie Mac Single-Family Loan-Level Dataset 
and verifies that all files are present and readable before any analysis begins.

The dataset is split by year and quarter (e.g., historical_data_2016Q1.txt).
Each year has two file types:
- Origination files: loan characteristics at the time the mortgage was issued
- Performance files: monthly payment history for each loan

Years used: 2016, 2017, 2018, 2019, 2020, 2021


# Quick Reload - Skip to Any Cell Without Rerunning

Run this single cell to restore all arrays, models, and metadata from disk.  
Skip all preprocessing and training cells entirely.

**Prerequisites:** The checkpoint save cell must have been run at least once.  
**Restores:** X_train, y_train, X_test, y_test, X_pre, X_post, shap_pre, shap_post,  
feature_cols, xgb_model, rf_model, sgd_model, scaler, encoders, fill_vals.

Use this as the starting point the notebook is reopen.

In [15]:
import numpy as np
import pandas as pd
import os
import joblib

# use absolute path directly
models_path = r"data\raw\models"
plots_path  = r"data\raw\models\plots"
data_path   = r"data\raw"
os.makedirs(plots_path, exist_ok=True)

# load all saved arrays
X_train      = np.load(os.path.join(models_path, "X_train.npy"))
y_train      = np.load(os.path.join(models_path, "y_train.npy"))
X_test       = np.load(os.path.join(models_path, "X_test.npy"))
y_test       = np.load(os.path.join(models_path, "y_test.npy"))
X_pre        = np.load(os.path.join(models_path, "X_pre.npy"))
X_post       = np.load(os.path.join(models_path, "X_post.npy"))
shap_pre     = np.load(os.path.join(models_path, "shap_pre.npy"))
shap_post    = np.load(os.path.join(models_path, "shap_post.npy"))
y_pre  = np.load(os.path.join(models_path, "y_pre.npy"))
y_post = np.load(os.path.join(models_path, "y_post.npy"))
feature_cols = pd.read_csv(os.path.join(models_path, "feature_cols.csv")).iloc[:, 0].tolist()

# load models
xgb_model = joblib.load(os.path.join(models_path, "xgb_model.pkl"))
rf_model  = joblib.load(os.path.join(models_path, "rf_model.pkl"))
sgd_model = joblib.load(os.path.join(models_path, "sgd_model.pkl"))
scaler    = joblib.load(os.path.join(models_path, "scaler.pkl"))
encoders  = joblib.load(os.path.join(models_path, "encoders.pkl"))
fill_vals = joblib.load(os.path.join(models_path, "col_fill_vals.pkl"))

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"X_pre:   {X_pre.shape}   | X_post: {X_post.shape}")
print(f"shap_pre: {shap_pre.shape} | shap_post: {shap_post.shape}")
print(f"Features: {len(feature_cols)}")
print("All arrays and models loaded. Ready to continue.")

X_train: (739002, 28) | X_test: (599841, 28)
X_pre:   (10000, 28)   | X_post: (10000, 28)
shap_pre: (10000, 28) | shap_post: (10000, 28)
Features: 28
All arrays and models loaded. Ready to continue.


In [18]:
import pandas as pd
import numpy as np
import os

# Define paths and time periods
years = [2016, 2017, 2018, 2019, 2020, 2021]
quarters = ['Q1', 'Q2', 'Q3', 'Q4']

# Create output folders if they don't exist
folders = [
    'data/processed',
    'data/samples',
    'outputs/models',
    'outputs/shap_values',
    'outputs/metrics',
    'outputs/figures'
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("ready")

ready


## Column Definitions

The Freddie Mac files have no header row.
Column names are assigned manually based on the official data dictionary.

Reference: Freddie Mac Single-Family Loan-Level Dataset User Guide
https://www.freddiemac.com/fmac-resources/research/pdf/user_guide.pdf
(Pages 5-14)


In [6]:
# Origination file - 32 columns 
orig_cols = [
    'credit_score',            # col 1
    'first_pmt_date',          # col 2
    'first_time_buyer',        # col 3
    'maturity_date',           # col 4
    'msa',                     # col 5
    'mi_pct',                  # col 6
    'num_units',               # col 7
    'occupancy_status',        # col 8
    'orig_cltv',               # col 9
    'orig_dti',                # col 10
    'orig_upb',                # col 11
    'orig_ltv',                # col 12
    'orig_interest_rate',      # col 13
    'channel',                 # col 14
    'prepay_penalty',          # col 15
    'amort_type',              # col 16
    'property_state',          # col 17
    'property_type',           # col 18
    'zip_3',                   # col 19
    'loan_seq_num',            # col 20
    'loan_purpose',            # col 21
    'orig_loan_term',          # col 22
    'num_borrowers',           # col 23
    'seller_name',             # col 24
    'servicer_name',           # col 25
    'super_conforming',        # col 26
    'pre_relief_seq_num',      # col 27
    'program_ind',             # col 28
    'relief_refinance_ind',    # col 29
    'property_val_method',     # col 30
    'io_indicator',            # col 31
    'mi_cancellation_ind'      # col 32
]

# Performance file - 32 columns 
perf_cols = [
    'loan_seq_num',                    # col 1
    'monthly_reporting_period',        # col 2
    'current_upb',                     # col 3
    'current_delinquency_status',      # col 4  KEY: used for target variable
    'loan_age',                        # col 5  KEY: used for target variable
    'remaining_months_to_maturity',    # col 6
    'defect_settlement_date',          # col 7
    'modification_flag',               # col 8
    'zero_balance_code',               # col 9  KEY: used for target variable
    'zero_balance_effective_date',     # col 10
    'current_interest_rate',           # col 11
    'current_deferred_upb',            # col 12
    'ddlpi',                           # col 13
    'mi_recoveries',                   # col 14
    'net_sales_proceeds',              # col 15
    'non_mi_recoveries',               # col 16
    'expenses',                        # col 17
    'legal_costs',                     # col 18
    'maintenance_costs',               # col 19
    'taxes_insurance',                 # col 20
    'misc_expenses',                   # col 21
    'actual_loss',                     # col 22
    'modification_cost',               # col 23
    'step_modification_flag',          # col 24
    'deferred_pmt_plan',               # col 25
    'estimated_ltv',                   # col 26
    'zero_balance_removal_upb',        # col 27
    'delinquent_accrued_int',          # col 28
    'delinquency_due_disaster',        # col 29
    'borrower_assistance_status',      # col 30
    'current_month_modification_loss', # col 31
    'interest_upb'                     # col 32
]

print(f"Origination columns: {len(orig_cols)}")   
print(f"Performance columns: {len(perf_cols)}")  


Origination columns: 32
Performance columns: 32


## File Verification

Before loading any data, we check that all expected files are present 
and note their sizes. This catches any missing or incomplete files/data.


In [19]:
print(f"{'File':<45} {'Orig Size':>12} {'Perf Size':>12}")
print("-" * 70)

missing = []

for year in years:
    for q in quarters:
        orig_file = os.path.join(data_path, f"historical_data_{year}{q}.txt")
        perf_file = os.path.join(data_path, f"historical_data_time_{year}{q}.txt")

        orig_exists = os.path.exists(orig_file)
        perf_exists = os.path.exists(perf_file)

        orig_size = f"{round(os.path.getsize(orig_file)/1024/1024, 1)} MB" if orig_exists else "MISSING"
        perf_size = f"{round(os.path.getsize(perf_file)/1024/1024, 1)} MB" if perf_exists else "MISSING"

        if not orig_exists:
            missing.append(orig_file)
        if not perf_exists:
            missing.append(perf_file)

        print(f"{year}{q:<41} {orig_size:>12} {perf_size:>12}")
if missing:
    print(f"\nMissing files ({len(missing)}):")
    for f in missing:
        print(f"  {f}")
else:
    print("\nAll files present.")

File                                             Orig Size    Perf Size
----------------------------------------------------------------------
2016Q1                                             44.6 MB    1633.5 MB
2016Q2                                             62.4 MB    2333.8 MB
2016Q3                                             69.2 MB    2618.5 MB
2016Q4                                             68.5 MB    2533.4 MB
2017Q1                                             42.4 MB    1354.8 MB
2017Q2                                             49.4 MB    1503.2 MB
2017Q3                                             57.3 MB    1751.0 MB
2017Q4                                             53.0 MB    1574.2 MB
2018Q1                                             43.7 MB    1140.5 MB
2018Q2                                             53.8 MB    1214.0 MB
2018Q3                                             49.5 MB    1045.1 MB
2018Q4                                             42.2 MB     80

## Loading Origination Files

Origination files contain one row per loan with static characteristics 
recorded at the time the mortgage was issued. These include credit score, 
loan-to-value ratio, debt-to-income ratio, loan amount, and property details.

All quarters per year are combined into a single dataframe.

In [6]:
def load_origination_year(year):
    frames = []
    for q in quarters:
        filepath = os.path.join(data_path, f"historical_data_{year}{q}.txt")
        if os.path.exists(filepath):
            df = pd.read_csv(
                filepath,
                sep='|',
                header=None,
                names=orig_cols,
                low_memory=False
            )
            df['orig_year'] = year
            df['orig_quarter'] = q
            frames.append(df)
            print(f"  {year}{q}: {df.shape[0]:,} loans loaded")
        else:
            print(f"  {year}{q}: file not found, skipping")
    return pd.concat(frames, ignore_index=True)


# Load all years
orig_frames = []
for year in years:
    print(f"\n{year}:")
    orig_frames.append(load_origination_year(year))

orig_all = pd.concat(orig_frames, ignore_index=True)

print(f"\nTotal origination records: {orig_all.shape[0]:,}")
print(f"Total columns: {orig_all.shape[1]}")


2016:
  2016Q1: 308,511 loans loaded
  2016Q2: 431,540 loans loaded
  2016Q3: 477,475 loans loaded
  2016Q4: 472,042 loans loaded

2017:
  2017Q1: 290,601 loans loaded
  2017Q2: 337,270 loans loaded
  2017Q3: 391,807 loans loaded
  2017Q4: 360,105 loans loaded

2018:
  2018Q1: 296,816 loans loaded
  2018Q2: 364,502 loans loaded
  2018Q3: 336,669 loans loaded
  2018Q4: 287,447 loans loaded

2019:
  2019Q1: 279,861 loans loaded
  2019Q2: 413,821 loans loaded
  2019Q3: 542,237 loans loaded
  2019Q4: 545,225 loans loaded

2020:
  2020Q1: 517,072 loans loaded
  2020Q2: 926,252 loans loaded
  2020Q3: 1,187,784 loans loaded
  2020Q4: 1,282,633 loans loaded

2021:
  2021Q1: 1,219,583 loans loaded
  2021Q2: 967,260 loans loaded
  2021Q3: 1,045,331 loans loaded
  2021Q4: 864,197 loans loaded

Total origination records: 14,146,041
Total columns: 34


## Building Labelled Dataset by Year by Year

Performance files are too large (-680M rows total) to load all at once.
Strategy: process one year at a time.
For each year:
1. Load that year's performance records (4 columns only)
2. Merge with that year's origination data
3. Compute the target variable: 1 if loan went 90+ DPD within 12 months, else 0
4. Keep only the final labelled rows — discard raw performance data
5. Save year's labelled data to parquet, free memory, move to next year

This keeps RAM usage less


In [7]:
import gc

perf_usecols = [0, 3, 4, 8]
perf_col_names = ['loan_seq_num', 'current_delinquency_status', 'loan_age', 'zero_balance_code']

output_path = os.path.join(data_path, "labelled")
os.makedirs(output_path, exist_ok=True)

def get_defaulted_loans_year(year):
    """
    Stream through all performance files for a year chunk by chunk.
    """
    defaulted = set()
    total_records = 0

    for q in quarters:
        filepath = os.path.join(data_path, f"historical_data_time_{year}{q}.txt")
        if not os.path.exists(filepath):
            print(f"  {year}{q}: not found, skipping")
            continue

        q_records = 0
        for chunk in pd.read_csv(
            filepath,
            sep='|',
            header=None,
            names=perf_cols,
            usecols=perf_usecols,
            low_memory=False,
            chunksize=300000,
            dtype={
                perf_cols[0]: str,
                perf_cols[3]: str,
                perf_cols[4]: 'Int16',
                perf_cols[8]: str
            }
        ):
            chunk.columns = perf_col_names
            chunk['current_delinquency_status'] = pd.to_numeric(
                chunk['current_delinquency_status'], errors='coerce'
            ).fillna(0)
            # Filter: within first 12 months AND 90+ DPD
            mask = (chunk['loan_age'] <= 12) & (chunk['current_delinquency_status'] >= 3)
            new_defaults = set(chunk.loc[mask, 'loan_seq_num'].unique())
            defaulted.update(new_defaults)
            q_records += len(chunk)
            del chunk
            gc.collect()

        total_records += q_records
        print(f"  {year}{q}: {q_records:,} records scanned, running defaults: {len(defaulted):,}")

    print(f"  Total records scanned: {total_records:,}")
    return defaulted


for year in years:
    print(f"\nProcessing {year}...")

    # Stream through performance — get defaulted loan IDs only
    defaulted_loans = get_defaulted_loans_year(year)
    print(f"  Defaulted loans found: {len(defaulted_loans):,}")

    # Get this year's origination data
    orig_year = orig_all[orig_all['orig_year'] == year].copy()

    # Assign target
    orig_year['target'] = orig_year['loan_seq_num'].isin(defaulted_loans).astype(int)

    default_rate = orig_year['target'].mean() * 100
    print(f"  Origination loans: {orig_year.shape[0]:,}")
    print(f"  Default rate: {default_rate:.2f}%")

    # Save to parquet
    out_file = os.path.join(output_path, f"labelled_{year}.parquet")
    orig_year.to_parquet(out_file, index=False)
    print(f"  Saved: {out_file}")

    del orig_year, defaulted_loans
    gc.collect()

print("\nAll years processed and saved.")
print(f"Files saved in: {output_path}")


Processing 2016...
  2016Q1: 20,029,514 records scanned, running defaults: 951
  2016Q2: 28,552,845 records scanned, running defaults: 2,202
  2016Q3: 31,989,541 records scanned, running defaults: 3,570
  2016Q4: 30,920,718 records scanned, running defaults: 5,830
  Total records scanned: 111,492,618
  Defaulted loans found: 5,830
  Origination loans: 1,689,568
  Default rate: 0.35%
  Saved: data/raw/labelled\labelled_2016.parquet

Processing 2017...
  2017Q1: 16,516,599 records scanned, running defaults: 2,019
  2017Q2: 18,296,657 records scanned, running defaults: 4,379
  2017Q3: 21,313,357 records scanned, running defaults: 6,461
  2017Q4: 19,164,748 records scanned, running defaults: 7,804
  Total records scanned: 75,291,361
  Defaulted loans found: 7,804
  Origination loans: 1,379,783
  Default rate: 0.57%
  Saved: data/raw/labelled\labelled_2017.parquet

Processing 2018...
  2018Q1: 13,873,084 records scanned, running defaults: 1,149
  2018Q2: 14,749,218 records scanned, running

## Load Labelled Data & Exploratory Data Analysis

All 6 labelled parquet files are loaded and combined into a single DataFrame.
We then examine:
- Overall class balance (default vs non-default)
- Default rates per year
- Missing value counts
- Feature distributions for key variables


In [8]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

output_path = os.path.join(data_path, "labelled")

# Load all 6 labelled parquet files
frames = []
for year in years:
    f = os.path.join(output_path, f"labelled_{year}.parquet")
    df = pd.read_parquet(f)
    frames.append(df)
    print(f"{year}: {df.shape[0]:,} rows, {df.shape[1]} columns, default rate: {df['target'].mean()*100:.2f}%")

full_df = pd.concat(frames, ignore_index=True)
print(f"\nFull dataset: {full_df.shape[0]:,} rows, {full_df.shape[1]} columns")
print(f"Overall default rate: {full_df['target'].mean()*100:.2f}%")
print(f"Defaults: {full_df['target'].sum():,} | Non-defaults: {(full_df['target']==0).sum():,}")

# Class imbalance ratio
ratio = (full_df['target']==0).sum() / full_df['target'].sum()
print(f"Class imbalance ratio: {ratio:.0f}:1 (non-default:default)")

# Missing values
print("\n--- Missing Values (top 15 columns) ---")
missing = full_df.isnull().sum()
missing_pct = (missing / len(full_df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False)
print(missing_df.head(15))

# Default rate by year
print("\n--- Default Rate by Year ---")
print(full_df.groupby('orig_year')['target'].agg(['sum','count','mean']).rename(
    columns={'sum':'defaults','count':'total','mean':'rate'}
).assign(rate=lambda x: (x['rate']*100).round(2)))

2016: 1,689,568 rows, 35 columns, default rate: 0.35%
2017: 1,379,783 rows, 35 columns, default rate: 0.57%
2018: 1,285,434 rows, 35 columns, default rate: 0.40%
2019: 1,781,144 rows, 35 columns, default rate: 2.72%
2020: 3,913,741 rows, 35 columns, default rate: 0.91%
2021: 4,096,371 rows, 35 columns, default rate: 0.46%

Full dataset: 14,146,041 rows, 35 columns
Overall default rate: 0.86%
Defaults: 121,713 | Non-defaults: 14,024,328
Class imbalance ratio: 115:1 (non-default:default)

--- Missing Values (top 15 columns) ---
                      missing_count  missing_pct
relief_refinance_ind       14001848        98.98
pre_relief_seq_num         14001848        98.98
super_conforming           13571800        95.94
msa                         1278279         9.04

--- Default Rate by Year ---
           defaults    total  rate
orig_year                         
2016           5830  1689568  0.35
2017           7804  1379783  0.57
2018           5079  1285434  0.40
2019          4846

## Preprocessing & Temporal Train/Test Split

Steps:
1. Drop columns with >90% missing values
2. Drop identifier columns not useful as features
3. Impute remaining missing values
4. Encode categorical columns
5. Temporal split: train = 2016–2019, test = 2020–2021
6. Save train and test sets to parquet


In [9]:
import pandas as pd
import numpy as np
import os, gc
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.preprocessing import LabelEncoder
from collections import defaultdict

output_path = os.path.join(data_path, "labelled")
drop_cols = ['loan_seq_num', 'relief_refinance_ind', 'pre_relief_seq_num',
             'super_conforming', 'orig_year', 'orig_quarter']
train_years = [2016, 2017, 2018, 2019]
test_years  = [2020, 2021]

# ─ PASS 1: Fit statistics from train only 
print("Pass 1: computing fit stats from train years...")
cat_modes   = {}
num_medians = {}
cat_uniques = defaultdict(set)   # for LabelEncoder
n_rows_seen = 0

for year in train_years:
    f = os.path.join(output_path, f"labelled_{year}.parquet")
    df = pd.read_parquet(f)
    df.drop(columns=drop_cols, inplace=True, errors='ignore')

    if n_rows_seen == 0:   # first year → determine column types
        cat_cols = [c for c in df.select_dtypes(include=['object']).columns if c != 'target']
        num_cols = [c for c in df.select_dtypes(include=['number']).columns if c != 'target']

    # Accumulate modes (use first-year mode as proxy)
    for col in cat_cols:
        if col not in cat_modes:
            cat_modes[col] = df[col].mode()[0]
        cat_uniques[col].update(df[col].dropna().astype(str).unique())

    # Accumulate running sum/count for median approximation
    # (exact median needs a second pass or use first-year median as proxy)
    for col in num_cols:
        if col not in num_medians:
            num_medians[col] = df[col].median()   # first-year proxy

    n_rows_seen += df.shape[0]
    print(f"  {year}: {df.shape[0]:,} rows scanned")
    del df; gc.collect()

# Fit LabelEncoders
print("Fitting LabelEncoders...")
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    le.fit(sorted(cat_uniques[col]))
    encoders[col] = le

# ─ PASS 2: Transform & stream-write each split 
def transform_and_write(year_list, out_path):
    pqwriter = None
    total_rows = 0
    for year in year_list:
        f = os.path.join(output_path, f"labelled_{year}.parquet")
        df = pd.read_parquet(f)
        df.drop(columns=drop_cols, inplace=True, errors='ignore')

        # Impute
        for col in cat_cols:
            df[col].fillna(cat_modes[col], inplace=True)
        for col in num_cols:
            df[col].fillna(num_medians[col], inplace=True)

        # Encode
        for col in cat_cols:
            le = encoders[col]
            df[col] = df[col].astype(str).apply(
                lambda x: x if x in le.classes_ else le.classes_[0]
            )
            df[col] = le.transform(df[col]).astype(np.int16)

        # Downcast numerics
        for col in num_cols:
            df[col] = pd.to_numeric(df[col], downcast='float')

        # Stream-write to parquet (no concat needed)
        table = pa.Table.from_pandas(df, preserve_index=False)
        if pqwriter is None:
            pqwriter = pq.ParquetWriter(out_path, table.schema, compression='snappy')
        pqwriter.write_table(table)
        total_rows += df.shape[0]
        print(f"  {year}: {df.shape[0]:,} rows written")
        del df, table; gc.collect()

    if pqwriter:
        pqwriter.close()
    return total_rows

print("\nPass 2: transforming & writing train.parquet...")
n = transform_and_write(train_years, os.path.join(output_path, "train.parquet"))
print(f"  → train.parquet: {n:,} total rows")

print("\nPass 2: transforming & writing test.parquet...")
n = transform_and_write(test_years, os.path.join(output_path, "test.parquet"))
print(f"  → test.parquet:  {n:,} total rows")

print("\nDone. Encoders and fit-stats available for inference.")

Pass 1: computing fit stats from train years...
  2016: 1,689,568 rows scanned
  2017: 1,379,783 rows scanned
  2018: 1,285,434 rows scanned
  2019: 1,781,144 rows scanned
Fitting LabelEncoders...

Pass 2: transforming & writing train.parquet...


C:\Users\sujit\AppData\Local\Temp\ipykernel_21720\546847325.py:66: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(cat_modes[col], inplace=True)
C:\Users\sujit\AppData\Local\Temp\ipykernel_21720\546847325.py:68: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example

  2016: 1,689,568 rows written


C:\Users\sujit\AppData\Local\Temp\ipykernel_21720\546847325.py:66: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(cat_modes[col], inplace=True)
C:\Users\sujit\AppData\Local\Temp\ipykernel_21720\546847325.py:68: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example

  2017: 1,379,783 rows written


...

  2019: 1,781,144 rows written
  → train.parquet: 6,135,929 total rows

Pass 2: transforming & writing test.parquet...


C:\Users\sujit\AppData\Local\Temp\ipykernel_21720\546847325.py:66: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(cat_modes[col], inplace=True)
C:\Users\sujit\AppData\Local\Temp\ipykernel_21720\546847325.py:68: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example

  2020: 3,913,741 rows written


C:\Users\sujit\AppData\Local\Temp\ipykernel_21720\546847325.py:66: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(cat_modes[col], inplace=True)
C:\Users\sujit\AppData\Local\Temp\ipykernel_21720\546847325.py:68: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example

  2021: 4,096,371 rows written
  → test.parquet:  8,010,112 total rows

Done. Encoders and fit-stats available for inference.


## Model Training

Three models are trained on the 2016–2019 training set:
- Logistic Regression (baseline)
- Random Forest
- XGBoost

All models use class_weight='balanced' (or scale_pos_weight for XGBoost)
to handle the 115:1 class imbalance without oversampling.

Training uses all 28 features. Target variable: binary default flag (1 = defaulted within 12 months).


In [10]:
import pandas as pd
import numpy as np
import os, gc
from sklearn.preprocessing import LabelEncoder

output_path = os.path.join(data_path, "labelled")
drop_cols = ['loan_seq_num', 'relief_refinance_ind', 'pre_relief_seq_num',
             'super_conforming', 'orig_year', 'orig_quarter']
train_years = [2016, 2017, 2018, 2019]
test_years  = [2020, 2021]
UNDERSAMPLE_RATIO = 10

# ─ Load & stratified undersample 
def load_sampled(year_list, label):
    frames = []
    for year in year_list:
        df = pd.read_parquet(os.path.join(output_path, f"labelled_{year}.parquet"))
        df.drop(columns=drop_cols, inplace=True, errors='ignore')
        defaults     = df[df['target'] == 1]
        non_defaults = df[df['target'] == 0].sample(
            n=min(len(defaults) * UNDERSAMPLE_RATIO, (df['target'] == 0).sum()),
            random_state=42
        )
        sampled = pd.concat([defaults, non_defaults]).sample(frac=1, random_state=42)
        frames.append(sampled)
        print(f"  {year}: {len(defaults):,} defaults + {len(non_defaults):,} non-defaults = {len(sampled):,} rows")
        del df, defaults, non_defaults, sampled
        gc.collect()
    out = pd.concat(frames, ignore_index=True)
    del frames; gc.collect()
    print(f"  → {label} total: {out.shape[0]:,} rows | Default rate: {out['target'].mean()*100:.2f}%\n")
    return out

print("Building stratified train sample...")
train_df = load_sampled(train_years, "Train")

print("Building stratified test sample...")
test_df = load_sampled(test_years, "Test")

# ─ Encode categoricals (fit on train only)
cat_cols = [c for c in train_df.select_dtypes(include=['object']).columns if c != 'target']
print(f"Encoding {len(cat_cols)} categorical columns: {cat_cols}")

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col].astype(str))
    test_df[col] = test_df[col].astype(str).apply(
        lambda x: x if x in le.classes_ else le.classes_[0]
    )
    test_df[col] = le.transform(test_df[col])
    encoders[col] = le

# ─ Impute numeric NaNs
num_cols = [c for c in train_df.select_dtypes(include=['number']).columns if c != 'target']
col_fill_vals = {}
for col in num_cols:
    median_val = train_df[col].median()
    train_df[col] = train_df[col].fillna(median_val)   # ← assignment, not inplace
    test_df[col]  = test_df[col].fillna(median_val)
    col_fill_vals[col] = median_val

# ─ Convert to numpy arrays 
feature_cols = [c for c in train_df.columns if c != 'target']
X_train = train_df[feature_cols].values.astype(np.float32)
y_train = train_df['target'].values
X_test  = test_df[feature_cols].values.astype(np.float32)
y_test  = test_df['target'].values
del train_df, test_df; gc.collect()

# ─numpy-level safety net for any NaN left
col_medians = np.nanmedian(X_train, axis=0)
for arr in [X_train, X_test]:
    bad_mask = ~np.isfinite(arr)
    if bad_mask.any():
        arr[bad_mask] = np.take(col_medians, np.where(bad_mask)[1])

assert np.isfinite(X_train).all(), "X_train still has NaN/inf!"
assert np.isfinite(X_test).all(),  "X_test still has NaN/inf!"
print(f" Arrays clean | X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"  Feature columns ({len(feature_cols)}): {feature_cols}")

Building stratified train sample...
  2016: 5,830 defaults + 58,300 non-defaults = 64,130 rows
  2017: 7,804 defaults + 78,040 non-defaults = 85,844 rows
  2018: 5,079 defaults + 50,790 non-defaults = 55,869 rows
  2019: 48,469 defaults + 484,690 non-defaults = 533,159 rows
  → Train total: 739,002 rows | Default rate: 9.09%

Building stratified test sample...
  2020: 35,573 defaults + 355,730 non-defaults = 391,303 rows
  2021: 18,958 defaults + 189,580 non-defaults = 208,538 rows
  → Test total: 599,841 rows | Default rate: 9.09%

Encoding 13 categorical columns: ['first_time_buyer', 'occupancy_status', 'channel', 'prepay_penalty', 'amort_type', 'property_state', 'property_type', 'loan_purpose', 'seller_name', 'servicer_name', 'program_ind', 'io_indicator', 'mi_cancellation_ind']
 Arrays clean | X_train: (739002, 28) | X_test: (599841, 28)
  Feature columns (28): ['credit_score', 'first_pmt_date', 'first_time_buyer', 'maturity_date', 'msa', 'mi_pct', 'num_units', 'occupancy_status', 

In [14]:
import joblib
import os

models_path = os.path.join(data_path, "models")
os.makedirs(models_path, exist_ok=True)

joblib.dump(encoders, os.path.join(models_path, "encoders.pkl"))
joblib.dump(col_fill_vals, os.path.join(models_path, "col_fill_vals.pkl"))
print("Saved encoders and fill values.")

Saved encoders and fill values.


In [15]:
import time, joblib
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             classification_report, precision_recall_curve)
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

models_path = os.path.join(data_path, "models")
os.makedirs(models_path, exist_ok=True)

results = {}

# --- 1. SGD Classifier (Logistic Regression baseline) ---

# SGD requires scaled features; Random Forest and XGBoost do not
print("[1/3] Scaling features for SGD...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
joblib.dump(scaler, os.path.join(models_path, "scaler.pkl"))

print("[1/3] Training SGD Classifier...")
t0 = time.time()
sgd = SGDClassifier(
    loss='log_loss',
    penalty='l2',
    class_weight='balanced',
    max_iter=100,
    tol=1e-3,
    n_jobs=-1,
    random_state=42,
    verbose=1
)
sgd.fit(X_train_scaled, y_train)
sgd_proba = sgd.predict_proba(X_test_scaled)[:, 1]

results['SGD (LR baseline)'] = {
    'auc_roc':       roc_auc_score(y_test, sgd_proba),
    'avg_precision': average_precision_score(y_test, sgd_proba),
    'report':        classification_report(y_test, (sgd_proba >= 0.5).astype(int))
}
print(f"  Done in {time.time()-t0:.1f}s | AUC-ROC: {results['SGD (LR baseline)']['auc_roc']:.4f}")
joblib.dump(sgd, os.path.join(models_path, "sgd_model.pkl"))
del sgd, sgd_proba, X_train_scaled, X_test_scaled
gc.collect()

# --- 2. Random Forest ---

print("\n[2/3] Training Random Forest...")
t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    max_depth=10,
    min_samples_leaf=50,
    n_jobs=-1,
    random_state=42,
    verbose=1
)
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]

# Find optimal classification threshold using F1 score on PR curve
prec, rec, thresholds = precision_recall_curve(y_test, rf_proba)
f1_scores = 2 * prec * rec / (prec + rec + 1e-9)
best_thresh_rf = thresholds[np.argmax(f1_scores)]
print(f"  Optimal threshold: {best_thresh_rf:.4f}")

results['Random Forest'] = {
    'auc_roc':       roc_auc_score(y_test, rf_proba),
    'avg_precision': average_precision_score(y_test, rf_proba),
    'report':        classification_report(y_test, (rf_proba >= best_thresh_rf).astype(int))
}
print(f"  Done in {time.time()-t0:.1f}s | AUC-ROC: {results['Random Forest']['auc_roc']:.4f}")
joblib.dump(rf, os.path.join(models_path, "rf_model.pkl"))
del rf, rf_proba
gc.collect()

# --- 3. XGBoost ---

print("\n[3/3] Training XGBoost...")
t0 = time.time()
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos,
    eval_metric='logloss',
    n_jobs=-1,
    random_state=42,
    tree_method='hist'
)
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=50)
xgb_proba = xgb.predict_proba(X_test)[:, 1]

prec, rec, thresholds = precision_recall_curve(y_test, xgb_proba)
f1_scores = 2 * prec * rec / (prec + rec + 1e-9)
best_thresh_xgb = thresholds[np.argmax(f1_scores)]
print(f"  Optimal threshold: {best_thresh_xgb:.4f}")

results['XGBoost'] = {
    'auc_roc':       roc_auc_score(y_test, xgb_proba),
    'avg_precision': average_precision_score(y_test, xgb_proba),
    'report':        classification_report(y_test, (xgb_proba >= best_thresh_xgb).astype(int))
}
print(f"  Done in {time.time()-t0:.1f}s | AUC-ROC: {results['XGBoost']['auc_roc']:.4f}")
joblib.dump(xgb, os.path.join(models_path, "xgb_model.pkl"))
del xgb, xgb_proba
gc.collect()

# --- Summary ---

print("\nMODEL COMPARISON")
for model, metrics in results.items():
    print(f"\n{model}:")
    print(f"  AUC-ROC:       {metrics['auc_roc']:.4f}")
    print(f"  Avg Precision: {metrics['avg_precision']:.4f}")
    print(metrics['report'])

[1/3] Scaling features for SGD...
[1/3] Training SGD Classifier...
-- Epoch 1
Norm: 3.83, NNZs: 25, Bias: -0.299059, T: 739002, Avg. loss: 1.809772
Total training time: 0.12 seconds.
-- Epoch 2
Norm: 2.28, NNZs: 25, Bias: -0.282547, T: 1478004, Avg. loss: 0.658636
Total training time: 0.24 seconds.
-- Epoch 3
Norm: 3.30, NNZs: 25, Bias: -0.365573, T: 2217006, Avg. loss: 0.637388
Total training time: 0.36 seconds.
-- Epoch 4
Norm: 2.87, NNZs: 25, Bias: -0.437708, T: 2956008, Avg. loss: 0.629565
Total training time: 0.48 seconds.
-- Epoch 5
Norm: 2.92, NNZs: 25, Bias: -0.341003, T: 3695010, Avg. loss: 0.624559
Total training time: 0.60 seconds.
-- Epoch 6
Norm: 2.79, NNZs: 25, Bias: -0.340993, T: 4434012, Avg. loss: 0.621691
Total training time: 0.71 seconds.
-- Epoch 7
Norm: 2.98, NNZs: 25, Bias: -0.369643, T: 5173014, Avg. loss: 0.620754
Total training time: 0.84 seconds.
-- Epoch 8
Norm: 2.00, NNZs: 25, Bias: -0.292466, T: 5912016, Avg. loss: 0.618279
Total training time: 0.95 seconds

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 32 concurrent workers.
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    8.5s finished
[Parallel(n_jobs=32)]: Using backend ThreadingBackend with 32 concurrent workers.
[Parallel(n_jobs=32)]: Done 100 out of 100 | elapsed:    0.3s finished


  Optimal threshold: 0.6193
  Done in 9.5s | AUC-ROC: 0.7533

[3/3] Training XGBoost...
[0]	validation_0-logloss:0.68471
[50]	validation_0-logloss:0.64032
[100]	validation_0-logloss:0.53239
[150]	validation_0-logloss:0.42698
[199]	validation_0-logloss:0.38818
  Optimal threshold: 0.4766
  Done in 10.0s | AUC-ROC: 0.7504

MODEL COMPARISON

SGD (LR baseline):
  AUC-ROC:       0.7189
  Avg Precision: 0.2003
              precision    recall  f1-score   support

           0       0.96      0.59      0.73    545310
           1       0.15      0.73      0.25     54531

    accuracy                           0.61    599841
   macro avg       0.55      0.66      0.49    599841
weighted avg       0.88      0.61      0.69    599841


Random Forest:
  AUC-ROC:       0.7533
  Avg Precision: 0.2304
              precision    recall  f1-score   support

           0       0.94      0.83      0.88    545310
           1       0.22      0.49      0.31     54531

    accuracy                         

# The below cell is a checkpoint cell to save model training in the memory/os to save time

In [31]:
import numpy as np
import os

models_path = os.path.join(data_path, "models")
os.makedirs(models_path, exist_ok=True)

# save training and test arrays so you never need to rebuild them again
np.save(os.path.join(models_path, "X_train.npy"), X_train)
np.save(os.path.join(models_path, "y_train.npy"), y_train)
np.save(os.path.join(models_path, "X_test.npy"),  X_test)
np.save(os.path.join(models_path, "y_test.npy"),  y_test)
print(f"Saved X_train {X_train.shape}, y_train {y_train.shape}")
print(f"Saved X_test  {X_test.shape},  y_test  {y_test.shape}")

Saved X_train (739002, 28), y_train (739002,)
Saved X_test  (599841, 28),  y_test  (599841,)


In [16]:
!pip install shap
!pip install joblib

In [17]:
import shap
import joblib
import numpy as np
import pandas as pd
import os, gc
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

output_path = os.path.join(data_path, "labelled")
models_path = os.path.join(data_path, "models")

drop_cols = ['loan_seq_num', 'relief_refinance_ind', 'pre_relief_seq_num',
             'super_conforming', 'orig_year', 'orig_quarter']

# Load model, encoders, and fill values from training
xgb          = joblib.load(os.path.join(models_path, "xgb_model.pkl"))
encoders     = joblib.load(os.path.join(models_path, "encoders.pkl"))
col_fill_vals = joblib.load(os.path.join(models_path, "col_fill_vals.pkl"))

print("Building SHAP samples...")

def build_shap_sample(years, label, n_per_class=5000):
    frames = []
    for year in years:
        df = pd.read_parquet(os.path.join(output_path, f"labelled_{year}.parquet"))
        df.drop(columns=drop_cols, inplace=True, errors='ignore')
        frames.append(df)
        del df; gc.collect()

    combined = pd.concat(frames, ignore_index=True)
    del frames; gc.collect()

    # Apply train encoders — unknown categories mapped to first known class
    for col, le in encoders.items():
        if col not in combined.columns:
            continue
        combined[col] = combined[col].astype(str).apply(
            lambda x: x if x in le.classes_ else le.classes_[0]
        )
        combined[col] = le.transform(combined[col])

    # Impute using train fill values — assignment, not inplace
    for col, fill_val in col_fill_vals.items():
        if col in combined.columns:
            combined[col] = combined[col].fillna(fill_val)

    n_defaults = int(combined['target'].sum())
    defaults     = combined[combined['target'] == 1].sample(
        n=min(n_per_class, n_defaults), random_state=42
    )
    non_defaults = combined[combined['target'] == 0].sample(
        n=n_per_class, random_state=42
    )
    sample = pd.concat([defaults, non_defaults]).sample(frac=1, random_state=42)
    print(f"  {label}: {len(sample):,} rows "
          f"({len(defaults):,} defaults + {len(non_defaults):,} non-defaults)")
    del combined, defaults, non_defaults; gc.collect()
    return sample


pre_covid  = build_shap_sample([2016, 2017, 2018, 2019], "Pre-COVID (train years)")
post_covid = build_shap_sample([2020, 2021],             "Post-COVID (test years)")

feature_cols = [c for c in pre_covid.columns if c != 'target']

X_pre  = pre_covid[feature_cols].values.astype(np.float32)
X_post = post_covid[feature_cols].values.astype(np.float32)
del pre_covid, post_covid; gc.collect()

# Numpy-level safety net for any residual NaN/inf
col_medians = np.nanmedian(X_pre, axis=0)
for arr in [X_pre, X_post]:
    bad_mask = ~np.isfinite(arr)
    if bad_mask.any():
        arr[bad_mask] = np.take(col_medians, np.where(bad_mask)[1])

assert np.isfinite(X_pre).all(),  "X_pre still has NaN/inf after cleaning."
assert np.isfinite(X_post).all(), "X_post still has NaN/inf after cleaning."

# Compute SHAP values using TreeExplainer
print("\nComputing SHAP values (pre-COVID)...")
explainer = shap.TreeExplainer(xgb)
shap_pre  = explainer.shap_values(X_pre)
print("Computing SHAP values (post-COVID)...")
shap_post = explainer.shap_values(X_post)

print("SHAP computation complete.")
print(f"  shap_pre shape:  {shap_pre.shape}")
print(f"  shap_post shape: {shap_post.shape}")

# Save all arrays
np.save(os.path.join(models_path, "shap_pre.npy"),  shap_pre)
np.save(os.path.join(models_path, "shap_post.npy"), shap_post)
np.save(os.path.join(models_path, "X_pre.npy"),     X_pre)
np.save(os.path.join(models_path, "X_post.npy"),    X_post)
pd.Series(feature_cols).to_csv(
    os.path.join(models_path, "feature_cols.csv"), index=False
)
print("All SHAP arrays saved.")

Building SHAP samples...
  Pre-COVID (train years): 10,000 rows (5,000 defaults + 5,000 non-defaults)
  Post-COVID (test years): 10,000 rows (5,000 defaults + 5,000 non-defaults)

Computing SHAP values (pre-COVID)...
Computing SHAP values (post-COVID)...
SHAP computation complete.
  shap_pre shape:  (10000, 28)
  shap_post shape: (10000, 28)
All SHAP arrays saved.


In [18]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

models_path = os.path.join(data_path, "models")
plots_path  = os.path.join(data_path, "plots")
os.makedirs(plots_path, exist_ok=True)

# Load everything
shap_pre  = np.load(os.path.join(models_path, "shap_pre.npy"))
shap_post = np.load(os.path.join(models_path, "shap_post.npy"))
X_pre     = np.load(os.path.join(models_path, "X_pre.npy"))
X_post    = np.load(os.path.join(models_path, "X_post.npy"))
feature_cols = pd.read_csv(os.path.join(models_path, "feature_cols.csv"))['0'].tolist()

print(f"Features: {feature_cols}")

# ─ Plot 1: SHAP Summary — Pre-COVID
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_pre, X_pre, feature_names=feature_cols, show=False, max_display=15)
plt.title("SHAP Feature Importance — Pre-COVID (2016–2019)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(plots_path, "shap_summary_pre.png"), dpi=150, bbox_inches='tight')
plt.close()
print("Saved: shap_summary_pre.png")

# ─ Plot 2: SHAP Summary — Post-COVID 
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_post, X_post, feature_names=feature_cols, show=False, max_display=15)
plt.title("SHAP Feature Importance — Post-COVID (2020–2021)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(plots_path, "shap_summary_post.png"), dpi=150, bbox_inches='tight')
plt.close()
print("Saved: shap_summary_post.png")

# ─ Plot 3: Feature Importance Shift (bar chart) 
mean_pre  = pd.Series(np.abs(shap_pre).mean(axis=0),  index=feature_cols).sort_values(ascending=False).head(15)
mean_post = pd.Series(np.abs(shap_post).mean(axis=0), index=feature_cols).sort_values(ascending=False).head(15)

all_features = list(set(mean_pre.index) | set(mean_post.index))
df_shift = pd.DataFrame({
    'Pre-COVID':  [mean_pre.get(f, 0)  for f in all_features],
    'Post-COVID': [mean_post.get(f, 0) for f in all_features]
}, index=all_features).sort_values('Pre-COVID', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
x = np.arange(len(df_shift))
width = 0.35
ax.barh(x - width/2, df_shift['Pre-COVID'],  width, label='Pre-COVID (2016–2019)', color='steelblue')
ax.barh(x + width/2, df_shift['Post-COVID'], width, label='Post-COVID (2020–2021)', color='coral')
ax.set_yticks(x)
ax.set_yticklabels(df_shift.index, fontsize=10)
ax.set_xlabel('Mean |SHAP Value|', fontsize=12)
ax.set_title('Feature Importance Shift: Pre-COVID vs Post-COVID', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(plots_path, "shap_importance_shift.png"), dpi=150, bbox_inches='tight')
plt.close()
print("Saved: shap_importance_shift.png")

print(f"\nAll plots saved to: {plots_path}")
print("\nTop 10 features pre-COVID:")
print(mean_pre.head(10))
print("\nTop 10 features post-COVID:")
print(mean_post.head(10))

Features: ['credit_score', 'first_pmt_date', 'first_time_buyer', 'maturity_date', 'msa', 'mi_pct', 'num_units', 'occupancy_status', 'orig_cltv', 'orig_dti', 'orig_upb', 'orig_ltv', 'orig_interest_rate', 'channel', 'prepay_penalty', 'amort_type', 'property_state', 'property_type', 'zip_3', 'loan_purpose', 'orig_loan_term', 'num_borrowers', 'seller_name', 'servicer_name', 'program_ind', 'property_val_method', 'io_indicator', 'mi_cancellation_ind']
Saved: shap_summary_pre.png
Saved: shap_summary_post.png
Saved: shap_importance_shift.png

All plots saved to: data/raw/plots

Top 10 features pre-COVID:
credit_score      0.585384
servicer_name     0.453545
first_pmt_date    0.418832
num_borrowers     0.298301
seller_name       0.284893
orig_dti          0.281887
orig_cltv         0.234112
maturity_date     0.189127
orig_upb          0.151888
zip_3             0.151551
dtype: float32

Top 10 features post-COVID:
first_pmt_date        0.531137
credit_score          0.507072
servicer_name       

In [19]:
# ─ SHAP Dependence Plots for top 4 features
top_features = ['credit_score', 'first_pmt_date', 'orig_dti', 'orig_cltv']

for feat in top_features:
    feat_idx = feature_cols.index(feat)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

    shap.dependence_plot(feat_idx, shap_pre, X_pre,
                         feature_names=feature_cols, ax=axes[0], show=False)
    axes[0].set_title(f'{feat} — Pre-COVID (2016–2019)', fontsize=12, fontweight='bold')

    shap.dependence_plot(feat_idx, shap_post, X_post,
                         feature_names=feature_cols, ax=axes[1], show=False)
    axes[1].set_title(f'{feat} — Post-COVID (2020–2021)', fontsize=12, fontweight='bold')

    plt.suptitle(f'SHAP Dependence: {feat}', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    fname = os.path.join(plots_path, f"shap_dependence_{feat}.png")
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: shap_dependence_{feat}.png")

print("\nAll dependence plots saved.")


Saved: shap_dependence_credit_score.png
Saved: shap_dependence_first_pmt_date.png
Saved: shap_dependence_orig_dti.png
Saved: shap_dependence_orig_cltv.png

All dependence plots saved.


In [6]:
# Cap extreme outliers in labelled parquet files 
import pandas as pd
import os

output_path = os.path.join(data_path, "labelled")

# Reasonable caps based on Freddie Mac data dictionary
caps = {
    'credit_score':      (300, 850),
    'orig_dti':          (0, 65),
    'orig_cltv':         (0, 105),
    'orig_ltv':          (0, 105),
    'orig_interest_rate': (0, 15),
    'orig_upb':          (0, 2000000),
    'orig_loan_term':    (60, 360),
}

for year in [2016, 2017, 2018, 2019, 2020, 2021]:
    f = os.path.join(output_path, f"labelled_{year}.parquet")
    df = pd.read_parquet(f)
    for col, (lo, hi) in caps.items():
        if col in df.columns:
            before = (df[col] < lo).sum() + (df[col] > hi).sum()
            df[col] = df[col].clip(lo, hi)
            if before > 0:
                print(f"  {year} {col}: clipped {before:,} values")
    df.to_parquet(f, index=False)
    print(f"  {year}: saved")

print("\nDone. Now re-run Cell 8 (model training) and Cell 9 (SHAP) again.")


  2016: saved
  2017: saved
  2018: saved
  2019: saved
  2020: saved
  2021: saved

Done. Now re-run Cell 8 (model training) and Cell 9 (SHAP) again.


In [21]:
# Regenerate dependence plots with proper axis limits 
top_features = ['credit_score', 'first_pmt_date', 'orig_dti', 'orig_cltv']

# Valid axis limits for pre-COVID plots
xlims_pre = {
    'credit_score':   (300, 850),
    'orig_dti':       (0, 65),
    'orig_cltv':      (0, 105),
    'first_pmt_date': (None, None)  # already clean
}

for feat in top_features:
    feat_idx = feature_cols.index(feat)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

    shap.dependence_plot(feat_idx, shap_pre, X_pre,
                         feature_names=feature_cols, ax=axes[0], show=False)
    axes[0].set_title(f'{feat} — Pre-COVID (2016–2019)', fontsize=12, fontweight='bold')
    
    # Apply x-axis limit to remove outliers visually
    lo, hi = xlims_pre.get(feat, (None, None))
    if lo is not None:
        axes[0].set_xlim(lo, hi)

    shap.dependence_plot(feat_idx, shap_post, X_post,
                         feature_names=feature_cols, ax=axes[1], show=False)
    axes[1].set_title(f'{feat} — Post-COVID (2020–2021)', fontsize=12, fontweight='bold')

    plt.suptitle(f'SHAP Dependence: {feat}', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    fname = os.path.join(plots_path, f"shap_dependence_{feat}_clean.png")
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: shap_dependence_{feat}_clean.png")

print("\nAll clean dependence plots saved.")


Saved: shap_dependence_credit_score_clean.png
Saved: shap_dependence_first_pmt_date_clean.png
Saved: shap_dependence_orig_dti_clean.png
Saved: shap_dependence_orig_cltv_clean.png

All clean dependence plots saved.


In [20]:
from scipy.stats import spearmanr
import numpy as np
import pandas as pd

# Mean absolute SHAP per feature, both periods
mean_pre  = np.abs(shap_pre).mean(axis=0)
mean_post = np.abs(shap_post).mean(axis=0)

# Spearman rank correlation of feature importance rankings
corr, pval = spearmanr(mean_pre, mean_post)

print(f"Spearman Rank Correlation (pre vs post COVID SHAP): {corr:.4f}")
print(f"P-value: {pval:.4f}")
print()

# Top 10 rank comparison table
df_ranks = pd.DataFrame({
    'feature':    feature_cols,
    'rank_pre':   pd.Series(mean_pre).rank(ascending=False).astype(int),
    'rank_post':  pd.Series(mean_post).rank(ascending=False).astype(int),
    'shap_pre':   mean_pre,
    'shap_post':  mean_post,
})
df_ranks['rank_change'] = df_ranks['rank_pre'] - df_ranks['rank_post']
df_ranks = df_ranks.sort_values('rank_pre')
print(df_ranks[['feature','rank_pre','rank_post','rank_change','shap_pre','shap_post']].to_string(index=False))


Spearman Rank Correlation (pre vs post COVID SHAP): 0.9441
P-value: 0.0000

            feature  rank_pre  rank_post  rank_change  shap_pre  shap_post
       credit_score         1          2           -1  0.585384   0.507072
      servicer_name         2          3           -1  0.453545   0.335476
     first_pmt_date         3          1            2  0.418832   0.531137
      num_borrowers         4         11           -7  0.298301   0.129209
        seller_name         5          5            0  0.284893   0.240484
           orig_dti         6          4            2  0.281887   0.329617
          orig_cltv         7          6            1  0.234112   0.208496
      maturity_date         8          7            1  0.189127   0.207974
           orig_upb         9          8            1  0.151888   0.177497
              zip_3        10         10            0  0.151551   0.153292
       loan_purpose        11         12           -1  0.121554   0.089159
 orig_interest_rate     

In [23]:
import numpy as np
import pandas as pd
import os, gc, time
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

models_path = os.path.join(data_path, "models")
plots_path  = os.path.join(data_path, "plots")
output_path = os.path.join(data_path, "labelled")

import shap, joblib
feature_cols = pd.read_csv(os.path.join(models_path, "feature_cols.csv"))['0'].tolist()

# ─ Build a small stratified sample for bootstrap experiments
print("Building bootstrap sample...")
frames = []
for year in [2016, 2017, 2018, 2019]:
    df = pd.read_parquet(os.path.join(output_path, f"labelled_{year}.parquet"))
    df.drop(columns=['loan_seq_num','relief_refinance_ind','pre_relief_seq_num',
                     'super_conforming','orig_year','orig_quarter'],
            inplace=True, errors='ignore')
    frames.append(df)
    del df; gc.collect()

base_df = pd.concat(frames, ignore_index=True)
del frames; gc.collect()

# Encode
cat_cols = [c for c in base_df.select_dtypes(include=['object']).columns if c != 'target']
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    base_df[col] = le.fit_transform(base_df[col].astype(str))
    encoders[col] = le

# Cap outliers
caps = {'credit_score':(300,850),'orig_dti':(0,65),'orig_cltv':(0,105),
        'orig_ltv':(0,105),'orig_interest_rate':(0,15),'orig_upb':(0,2000000)}
for col,(lo,hi) in caps.items():
    if col in base_df.columns:
        base_df[col] = base_df[col].clip(lo,hi)

num_cols = [c for c in base_df.select_dtypes(include=['number']).columns if c != 'target']
for col in num_cols:
    base_df[col].fillna(base_df[col].median(), inplace=True)

# Stratified sample: 2500 defaults + 2500 non-defaults
defaults     = base_df[base_df['target']==1].sample(n=2500, random_state=42)
non_defaults = base_df[base_df['target']==0].sample(n=2500, random_state=42)
boot_df = pd.concat([defaults, non_defaults]).sample(frac=1, random_state=42)
del base_df, defaults, non_defaults; gc.collect()

X_boot = boot_df[feature_cols].values.astype(np.float32)
y_boot = boot_df['target'].values
del boot_df; gc.collect()
print(f"Bootstrap base sample: {X_boot.shape}")

# ─ Run 30 bootstrap iterations
N_BOOTSTRAP = 30
scale = (y_boot == 0).sum() / (y_boot == 1).sum()
all_shap_importance = []

print(f"\nRunning {N_BOOTSTRAP} bootstrap iterations...")
t_start = time.time()

for i in range(N_BOOTSTRAP):
    # Resample with replacement
    idx = np.random.choice(len(X_boot), size=len(X_boot), replace=True)
    oob = np.setdiff1d(np.arange(len(X_boot)), idx)  # out-of-bag
    if len(oob) < 100:
        continue

    X_tr, y_tr = X_boot[idx], y_boot[idx]
    X_oob, y_oob = X_boot[oob], y_boot[oob]

    # Train XGBoost
    model = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                          scale_pos_weight=scale, eval_metric='logloss',
                          n_jobs=-1, random_state=i, tree_method='hist',
                          verbosity=0)
    model.fit(X_tr, y_tr)

    # SHAP on OOB
    explainer = shap.TreeExplainer(model)
    shap_vals = explainer.shap_values(X_oob)
    mean_imp  = np.abs(shap_vals).mean(axis=0)
    all_shap_importance.append(mean_imp)

    if (i+1) % 5 == 0:
        elapsed = time.time() - t_start
        print(f"  Iteration {i+1}/{N_BOOTSTRAP} | {elapsed:.0f}s elapsed | "
              f"~{elapsed/(i+1)*(N_BOOTSTRAP-i-1):.0f}s remaining")

all_shap_importance = np.array(all_shap_importance)  # shape: (30, 28)
print(f"\nBootstrap complete. Shape: {all_shap_importance.shape}")

# ─ Compute Kendall's W
from scipy.stats import kendalltau

# Rank each bootstrap run's feature importances
ranks = np.array([pd.Series(row).rank(ascending=False).values 
                  for row in all_shap_importance])

n = ranks.shape[0]  # number of raters (bootstrap runs)
k = ranks.shape[1]  # number of items (features)

# Kendall's W formula
R = ranks.sum(axis=0)           # sum of ranks per feature
mean_R = R.mean()
S = np.sum((R - mean_R)**2)
W = 12 * S / (n**2 * (k**3 - k))

print(f"\nKendall's W (bootstrap stability): {W:.4f}")
print("Interpretation: >0.8 = strong stability | 0.5-0.8 = moderate | <0.5 = unstable")

# ─ SHAP variance per feature 
shap_std  = all_shap_importance.std(axis=0)
shap_mean = all_shap_importance.mean(axis=0)
shap_cv   = shap_std / (shap_mean + 1e-9)  # coefficient of variation

df_stability = pd.DataFrame({
    'feature':   feature_cols,
    'mean_shap': shap_mean,
    'std_shap':  shap_std,
    'cv':        shap_cv,
    'rank':      pd.Series(shap_mean).rank(ascending=False).astype(int)
}).sort_values('mean_shap', ascending=False)

print("\nTop 15 Features — Bootstrap Stability:")
print(df_stability[['feature','mean_shap','std_shap','cv','rank']].head(15).to_string(index=False))

# ─ Plot: mean SHAP +/- std 
top15 = df_stability.head(15)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top15['feature'][::-1], top15['mean_shap'][::-1],
        xerr=top15['std_shap'][::-1], capsize=4,
        color='steelblue', alpha=0.8, error_kw={'ecolor':'black','linewidth':1.5})
ax.set_xlabel('Mean |SHAP Value| ± Std Dev (30 bootstrap runs)', fontsize=11)
ax.set_title(f"Bootstrap SHAP Stability (Kendall's W = {W:.4f})", 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(plots_path, "bootstrap_stability.png"), dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: bootstrap_stability.png")

# Save for next cell
np.save(os.path.join(models_path, "bootstrap_shap_importance.npy"), all_shap_importance)
print(f"Kendall's W = {W:.4f}")

Building bootstrap sample...
Bootstrap base sample: (5000, 28)

Running 30 bootstrap iterations...
  Iteration 5/30 | 1s elapsed | ~5s remaining
  Iteration 10/30 | 2s elapsed | ~4s remaining
  Iteration 15/30 | 3s elapsed | ~3s remaining
  Iteration 20/30 | 4s elapsed | ~2s remaining
  Iteration 25/30 | 4s elapsed | ~1s remaining
  Iteration 30/30 | 5s elapsed | ~0s remaining

Bootstrap complete. Shape: (30, 28)

Kendall's W (bootstrap stability): 0.9671
Interpretation: >0.8 = strong stability | 0.5-0.8 = moderate | <0.5 = unstable

Top 15 Features — Bootstrap Stability:
           feature  mean_shap  std_shap       cv  rank
    first_pmt_date   1.195040  0.051213 0.042855     1
      credit_score   0.673284  0.039664 0.058911     2
     servicer_name   0.325499  0.037382 0.114844     3
          orig_dti   0.298808  0.030965 0.103628     4
     num_borrowers   0.289305  0.033181 0.114691     5
          orig_upb   0.246195  0.027209 0.110520     6
         orig_cltv   0.163971  0.031

In [24]:
import numpy as np
import pandas as pd
import os, gc
from xgboost import XGBClassifier
from scipy.stats import spearmanr
import shap, matplotlib.pyplot as plt

models_path = os.path.join(data_path, "models")
plots_path  = os.path.join(data_path, "plots")

feature_cols = pd.read_csv(os.path.join(models_path, "feature_cols.csv"))['0'].tolist()

# Load the bootstrap base arrays (already in memory from Cell 13)
# X_boot and y_boot should still be in memory — if not, rebuild them
try:
    _ = X_boot.shape
    print(f"Using existing X_boot: {X_boot.shape}")
except NameError:
    print("X_boot not in memory — re-run Cell 13 first")

# ─ Train a clean baseline model on X_boot 
scale = (y_boot == 0).sum() / (y_boot == 1).sum()
baseline_model = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                                scale_pos_weight=scale, eval_metric='logloss',
                                n_jobs=-1, random_state=42, tree_method='hist',
                                verbosity=0)
baseline_model.fit(X_boot, y_boot)
explainer_base = shap.TreeExplainer(baseline_model)
shap_baseline  = explainer_base.shap_values(X_boot)
importance_base = np.abs(shap_baseline).mean(axis=0)
print("Baseline SHAP computed.")

# ─ Perturbation levels
noise_levels = [0.05, 0.10, 0.15]
results_perturb = {}

# Only perturb continuous (numeric) columns
cont_idx = [i for i, col in enumerate(feature_cols)
            if col in ['credit_score','orig_dti','orig_cltv','orig_ltv',
                       'orig_interest_rate','orig_upb','orig_loan_term']]
feat_stds = X_boot[:, cont_idx].std(axis=0)

for noise_pct in noise_levels:
    print(f"\nPerturbation level: {int(noise_pct*100)}%...")
    
    X_perturbed = X_boot.copy()
    for j, idx in enumerate(cont_idx):
        noise = np.random.normal(0, noise_pct * feat_stds[j], size=len(X_boot))
        X_perturbed[:, idx] += noise.astype(np.float32)

    # Train on perturbed data
    pert_model = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                                scale_pos_weight=scale, eval_metric='logloss',
                                n_jobs=-1, random_state=42, tree_method='hist',
                                verbosity=0)
    pert_model.fit(X_perturbed, y_boot)

    # SHAP on perturbed
    explainer_pert = shap.TreeExplainer(pert_model)
    shap_pert      = explainer_pert.shap_values(X_perturbed)
    importance_pert = np.abs(shap_pert).mean(axis=0)

    # Spearman rank correlation vs baseline
    corr, pval = spearmanr(importance_base, importance_pert)

    # Top-5 agreement
    top5_base = set(np.argsort(importance_base)[-5:])
    top5_pert = set(np.argsort(importance_pert)[-5:])
    top5_agree = len(top5_base & top5_pert) / 5

    results_perturb[noise_pct] = {
        'spearman': corr,
        'top5_agreement': top5_agree,
        'importance': importance_pert
    }
    print(f"  Spearman corr vs baseline: {corr:.4f} | Top-5 agreement: {top5_agree:.0%}")

# ─ Plot: Spearman correlation vs noise level 
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

noise_pcts = [int(n*100) for n in noise_levels]
spearman_vals = [results_perturb[n]['spearman'] for n in noise_levels]
top5_vals     = [results_perturb[n]['top5_agreement'] for n in noise_levels]

axes[0].plot([0]+noise_pcts, [1.0]+spearman_vals, 'o-', color='steelblue', linewidth=2, markersize=8)
axes[0].set_xlabel('Noise Level (% of feature std)', fontsize=11)
axes[0].set_ylabel('Spearman Rank Correlation', fontsize=11)
axes[0].set_title('SHAP Ranking Stability Under Perturbation', fontsize=12, fontweight='bold')
axes[0].set_ylim(0.7, 1.05)
axes[0].axhline(0.9, color='red', linestyle='--', alpha=0.5, label='0.9 threshold')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].bar(noise_pcts, top5_vals, color='coral', alpha=0.8, width=3)
axes[1].set_xlabel('Noise Level (% of feature std)', fontsize=11)
axes[1].set_ylabel('Top-5 Feature Agreement', fontsize=11)
axes[1].set_title('Top-5 Feature Overlap vs Baseline', fontsize=12, fontweight='bold')
axes[1].set_ylim(0, 1.1)
for i, v in enumerate(top5_vals):
    axes[1].text(noise_pcts[i], v+0.02, f'{v:.0%}', ha='center', fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3, axis='y')

plt.suptitle('Perturbation Sensitivity Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(plots_path, "perturbation_stability.png"), dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: perturbation_stability.png")

# Summary table
print("PERTURBATION SUMMARY")
print(f"{'Noise Level':<15} {'Spearman':<12} {'Top-5 Agreement'}")
print(f"{'Baseline (0%)':<15} {'1.0000':<12} {'100%'}")
for n in noise_levels:
    r = results_perturb[n]
    print(f"{int(n*100):<14}% {r['spearman']:<12.4f} {r['top5_agreement']:.0%}")

Using existing X_boot: (5000, 28)
Baseline SHAP computed.

Perturbation level: 5%...
  Spearman corr vs baseline: 0.9526 | Top-5 agreement: 100%

Perturbation level: 10%...
  Spearman corr vs baseline: 0.9485 | Top-5 agreement: 100%

Perturbation level: 15%...
  Spearman corr vs baseline: 0.9408 | Top-5 agreement: 100%

Saved: perturbation_stability.png
PERTURBATION SUMMARY
Noise Level     Spearman     Top-5 Agreement
Baseline (0%)   1.0000       100%
5             % 0.9526       100%
10            % 0.9485       100%
15            % 0.9408       100%


In [2]:
import pandas as pd
import numpy as np
import os

models_path = os.path.join(data_path, "models")
plots_path  = os.path.join(data_path, "plots")

feature_cols = pd.read_csv(os.path.join(models_path, "feature_cols.csv"))['0'].tolist()

#  Complete Results Summary
print("COMPLETE RESULTS SUMMARY ")

print("\n─ 1. MODEL PERFORMANCE ")
performance = {
    'SGD (Logistic Regression)': {'AUC-ROC': 0.7189, 'Avg Precision': 0.2033},
    'Random Forest':             {'AUC-ROC': 0.7533, 'Avg Precision': 0.2304},
    'XGBoost (primary)':         {'AUC-ROC': 0.7504, 'Avg Precision': 0.2615},
}
for model, metrics in performance.items():
    print(f"  {model:<30} AUC-ROC: {metrics['AUC-ROC']:.4f} | Avg Precision: {metrics['Avg Precision']:.4f}")

print("\n─ 2. TEMPORAL DRIFT (Pre vs Post COVID)")
print(f"  Spearman Rank Correlation:  0.9447  (p < 0.0001)")
print(f"  Interpretation:             Strong structural preservation with localised drift")
print(f"  Biggest risers:             first_pmt_date (+2 ranks), orig_dti (+2), seller_name (+2)")
print(f"  Biggest fallers:            num_borrowers (-6), servicer_name (-3), mi_cancellation_ind (-6)")

print("\n─ 3. BOOTSTRAP STABILITY (Within Pre-COVID Period) ")
print(f"  Kendall's W:                0.9663  (30 bootstrap runs, n=5,000)")
print(f"  Threshold for strong:       >0.8")
print(f"  Verdict:                    STRONGLY STABLE")
print(f"  Most stable feature:        first_pmt_date  (CV = 0.043)")
print(f"  Least stable (top 15):      channel         (CV = 0.463)")

print("\n─ 4. PERTURBATION ROBUSTNESS ")
print(f"  {'Noise Level':<15} {'Spearman':<12} {'Top-5 Agreement'}")
print(f"  {'Baseline (0%)':<15} {'1.0000':<12} {'100%'}")
print(f"  {'5%':<15} {'0.9347':<12} {'100%'}")
print(f"  {'10%':<15} {'0.9545':<12} {'100%'}")
print(f"  {'15%':<15} {'0.9370':<12} {'100%'}")
print(f"  Verdict:                    ROBUST to data quality noise")

# Save summary to CSV
summary_data = {
    'Analysis': ['Model Performance','Model Performance','Model Performance',
                 'Temporal Drift','Bootstrap Stability','Perturbation 5%',
                 'Perturbation 10%','Perturbation 15%'],
    'Metric':   ['AUC-ROC (SGD)','AUC-ROC (RF)','AUC-ROC (XGBoost)',
                 'Spearman Rank Corr','Kendall W',
                 'Spearman vs Baseline','Spearman vs Baseline','Spearman vs Baseline'],
    'Value':    [0.5095, 0.7533, 0.7504, 0.9447, 0.9663, 0.9347, 0.9545, 0.9370],
    'Verdict':  ['Baseline (non-linear confirmed)','Good','Good (primary model)',
                 'Strong preservation + localised drift','Strongly stable',
                 'Robust (100% top-5 agreement)','Robust (100% top-5 agreement)',
                 'Robust (100% top-5 agreement)']
}
pd.DataFrame(summary_data).to_csv(os.path.join(plots_path, "final_results_summary.csv"), index=False)
print("\nSaved: final_results_summary.csv")

COMPLETE RESULTS SUMMARY 

─ 1. MODEL PERFORMANCE 
  SGD (Logistic Regression)      AUC-ROC: 0.7189 | Avg Precision: 0.2033
  Random Forest                  AUC-ROC: 0.7533 | Avg Precision: 0.2304
  XGBoost (primary)              AUC-ROC: 0.7504 | Avg Precision: 0.2615

─ 2. TEMPORAL DRIFT (Pre vs Post COVID)
  Spearman Rank Correlation:  0.9447  (p < 0.0001)
  Interpretation:             Strong structural preservation with localised drift
  Biggest risers:             first_pmt_date (+2 ranks), orig_dti (+2), seller_name (+2)
  Biggest fallers:            num_borrowers (-6), servicer_name (-3), mi_cancellation_ind (-6)

─ 3. BOOTSTRAP STABILITY (Within Pre-COVID Period) 
  Kendall's W:                0.9663  (30 bootstrap runs, n=5,000)
  Threshold for strong:       >0.8
  Verdict:                    STRONGLY STABLE
  Most stable feature:        first_pmt_date  (CV = 0.043)
  Least stable (top 15):      channel         (CV = 0.463)

─ 4. PERTURBATION ROBUSTNESS 
  Noise Level     Spea

# KEY FINDINGS
1 SHAP explanations are STABLE within periods (W=0.966)and ROBUST to perturbations (Spearman>0.93) but show MEANINGFUL TEMPORAL DRIFT across COVID boundary (Spearman=0.945 with key rank shifts in 6 features).
Explanations are trustworthy for deployment BUT require periodic recalibration after macro shocks")



# Saving y Labels for Pre/Post Samples

In [15]:
import numpy as np
import pandas as pd
import os, gc
import joblib

models_path = os.path.join(data_path, "models")
output_path = os.path.join(data_path, "labelled")
drop_cols   = ['loan_seq_num', 'relief_refinance_ind', 'pre_relief_seq_num',
               'super_conforming', 'orig_year', 'orig_quarter']

y_pre_path  = os.path.join(models_path, "y_pre.npy")
y_post_path = os.path.join(models_path, "y_post.npy")

# Skip if already saved
if os.path.exists(y_pre_path) and os.path.exists(y_post_path):
    y_pre  = np.load(y_pre_path)
    y_post = np.load(y_post_path)
    print(f"y_pre and y_post already exist on disk — loaded in seconds.")
    print(f"y_pre: {y_pre.shape} | y_post: {y_post.shape}")

else:
    print("Files not found — building from parquet (first run on this machine)...")
    encoders  = joblib.load(os.path.join(models_path, "encoders.pkl"))
    fill_vals = joblib.load(os.path.join(models_path, "col_fill_vals.pkl"))

    def build_sample_with_labels(years, label, n_per_class=5000):
        frames = []
        for year in years:
            df = pd.read_parquet(os.path.join(output_path, f"labelled_{year}.parquet"))
            df.drop(columns=drop_cols, inplace=True, errors='ignore')
            frames.append(df)
            del df
            gc.collect()
        combined = pd.concat(frames, ignore_index=True)
        del frames
        gc.collect()
        for col, le in encoders.items():
            if col not in combined.columns:
                continue
            combined[col] = combined[col].astype(str).apply(
                lambda x: x if x in le.classes_ else le.classes_[0])
            combined[col] = le.transform(combined[col])
        for col, fv in fill_vals.items():
            if col in combined.columns:
                combined[col] = combined[col].fillna(fv)
        n_defaults   = int(combined['target'].sum())
        defaults     = combined[combined['target'] == 1].sample(n=min(n_per_class, n_defaults), random_state=42)
        non_defaults = combined[combined['target'] == 0].sample(n=n_per_class, random_state=42)
        sample       = pd.concat([defaults, non_defaults]).sample(frac=1, random_state=42)
        del combined, defaults, non_defaults
        gc.collect()
        print(f"  {label}: {len(sample)} rows")
        return sample

    feature_cols = pd.read_csv(os.path.join(models_path, "feature_cols.csv")).iloc[:, 0].tolist()
    pre_sample   = build_sample_with_labels([2016, 2017, 2018, 2019], "Pre-COVID")
    post_sample  = build_sample_with_labels([2020, 2021], "Post-COVID")
    y_pre        = pre_sample['target'].values
    y_post       = post_sample['target'].values
    np.save(y_pre_path,  y_pre)
    np.save(y_post_path, y_post)
    print("Saved y_pre.npy and y_post.npy")

print("Done")

y_pre and y_post already exist on disk — loaded in seconds.
y_pre: (10000,) | y_post: (10000,)
Done


# SMOTE vs Class-Weighting SHAP Comparison

In [25]:
import sys
!{sys.executable} -m pip install imbalanced-learn

  Using cached imbalanced_learn-0.12.4-py3-none-any.whl.metadata (8.3 kB)
Using cached imbalanced_learn-0.12.4-py3-none-any.whl (258 kB)



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: C:\Users\sujit\AppData\Local\Programs\Python\Python39\python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import os, gc, time
import joblib
import shap
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from scipy.stats import spearmanr

models_path = os.path.join(data_path, "models")
plots_path  = os.path.join(data_path, "plots")
os.makedirs(plots_path, exist_ok=True)

# Apply SMOTE to training data only
print("Applying SMOTE to training set...")
t0 = time.time()
sm = SMOTE(sampling_strategy=0.3, random_state=42, n_jobs=-1)
X_smote, y_smote = sm.fit_resample(X_train, y_train)
print(f"  Done in {time.time()-t0:.1f}s | shape: {X_smote.shape} | default rate: {y_smote.mean()*100:.2f}%")

# Train XGBoost on SMOTE-resampled data
print("\nTraining XGBoost (SMOTE)...")
t0 = time.time()
xgb_smote = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    eval_metric='logloss', n_jobs=-1, random_state=42,
    tree_method='hist', verbosity=0
)
xgb_smote.fit(X_smote, y_smote)
print(f"  Done in {time.time()-t0:.1f}s")
del X_smote, y_smote
gc.collect()

# Compute SHAP for both models on same post-COVID sample
print("\nComputing SHAP values...")
exp_cw    = shap.TreeExplainer(xgb_model)
exp_smote = shap.TreeExplainer(xgb_smote)
shap_cw    = exp_cw.shap_values(X_post)
shap_smote = exp_smote.shap_values(X_post)

imp_cw    = np.abs(shap_cw).mean(axis=0)
imp_smote = np.abs(shap_smote).mean(axis=0)

corr, pval   = spearmanr(imp_cw, imp_smote)
top5_cw      = set(np.argsort(imp_cw)[-5:])
top5_smote   = set(np.argsort(imp_smote)[-5:])
top5_overlap = len(top5_cw & top5_smote)

print(f"\nSpearman rank correlation: {corr:.4f}  (p = {pval:.6f})")
print(f"Top-5 feature agreement:   {top5_overlap}/5")

# Rank comparison table
df_compare = pd.DataFrame({
    'feature':    feature_cols,
    'shap_cw':    imp_cw,
    'shap_smote': imp_smote,
    'rank_cw':    pd.Series(imp_cw).rank(ascending=False).astype(int),
    'rank_smote': pd.Series(imp_smote).rank(ascending=False).astype(int),
})
df_compare['rank_diff'] = (df_compare['rank_cw'] - df_compare['rank_smote']).abs()
df_compare = df_compare.sort_values('rank_cw').reset_index(drop=True)

print("\nTop 15 features:")
print(df_compare[['feature','rank_cw','rank_smote','rank_diff','shap_cw','shap_smote']].head(15).to_string(index=False))

# Bar chart
top15 = df_compare.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 7))
x = np.arange(len(top15))
w = 0.35
ax.barh(x - w/2, top15['shap_cw'],    w, label='Class-Weighted', color='steelblue', alpha=0.85)
ax.barh(x + w/2, top15['shap_smote'], w, label='SMOTE',          color='coral',     alpha=0.85)
ax.set_yticks(x)
ax.set_yticklabels(top15['feature'], fontsize=10)
ax.set_xlabel('Mean |SHAP| Value', fontsize=11)
ax.set_title('SMOTE vs Class-Weighting: SHAP Feature Importance', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(os.path.join(plots_path, "smote_vs_cw_shap.png"), dpi=150, bbox_inches='tight')
plt.close()

# Save outputs
np.save(os.path.join(models_path, "shap_smote.npy"), shap_smote)
joblib.dump(xgb_smote, os.path.join(models_path, "xgb_smote_model.pkl"))
df_compare.to_csv(os.path.join(plots_path, "smote_vs_cw_ranks.csv"), index=False)
print("\nSaved smote_vs_cw_shap.png and smote_vs_cw_ranks.csv")

C:\Users\sujit\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Applying SMOTE to training set...


C:\Users\sujit\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
C:\Users\sujit\AppData\Local\Programs\Python\Python39\lib\site-packages\imblearn\over_sampling\_smote\base.py:370: FutureWarning: The parameter `n_jobs` has been deprecated in 0.10 and will be removed in 0.12. You can pass an nearest neighbors estimator where `n_jobs` is already set instead.
  warnings.warn(


  Done in 3.3s | shape: (873366, 28) | default rate: 23.08%

Training XGBoost (SMOTE)...
  Done in 2.8s

Computing SHAP values...

Spearman rank correlation: 0.7742  (p = 0.000001)
Top-5 feature agreement:   3/5

Top 15 features:
           feature  rank_cw  rank_smote  rank_diff  shap_cw  shap_smote
    first_pmt_date        1           9          8 0.531137    0.178533
      credit_score        2           2          0 0.507072    0.555269
     servicer_name        3           4          1 0.335476    0.392392
          orig_dti        4           5          1 0.329617    0.281335
       seller_name        5           6          1 0.240484    0.266606
         orig_cltv        6          12          6 0.208496    0.141921
     maturity_date        7           3          4 0.207974    0.423265
          orig_upb        8           8          0 0.177497    0.205622
orig_interest_rate        9          16          7 0.165253    0.084649
             zip_3       10          14          4

# Interpretability-Performance Tradeoff

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, shap, os
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

models_path = os.path.join(data_path, "models")
plots_path  = os.path.join(data_path, "plots")

# Load pre/post labels
y_pre  = np.load(os.path.join(models_path, "y_pre.npy"))
y_post = np.load(os.path.join(models_path, "y_post.npy"))

X_pre_scaled  = scaler.transform(X_pre)
X_post_scaled = scaler.transform(X_post)

# AUC on pre and post samples for all 3 models
model_specs = [
    ("Logistic Regression", sgd_model, X_pre_scaled, X_post_scaled),
    ("Random Forest",       rf_model,  X_pre,        X_post),
    ("XGBoost",             xgb_model, X_pre,        X_post),
]

auc_results = {}
for name, mdl, Xp, Xq in model_specs:
    auc_pre  = roc_auc_score(y_pre,  mdl.predict_proba(Xp)[:, 1])
    auc_post = roc_auc_score(y_post, mdl.predict_proba(Xq)[:, 1])
    auc_results[name] = {
        'auc_pre':  auc_pre,
        'auc_post': auc_post,
        'auc_drop': auc_pre - auc_post,
    }
    print(f"{name:25s}  pre: {auc_pre:.4f}  post: {auc_post:.4f}  drop: {auc_pre-auc_post:.4f}")

# SHAP stability for all 3 models
def shap_spearman(sv_pre, sv_post):
    corr, _ = spearmanr(np.abs(sv_pre).mean(axis=0), np.abs(sv_post).mean(axis=0))
    return corr

print("\nComputing RF SHAP values...")
exp_rf     = shap.TreeExplainer(rf_model)
sv_rf_pre  = exp_rf.shap_values(X_pre)
sv_rf_post = exp_rf.shap_values(X_post)
if isinstance(sv_rf_pre, list):
    sv_rf_pre, sv_rf_post = sv_rf_pre[1], sv_rf_post[1]

print("Computing SGD SHAP values...")
exp_sgd     = shap.LinearExplainer(sgd_model, X_pre_scaled)
sv_sgd_pre  = exp_sgd.shap_values(X_pre_scaled)
sv_sgd_post = exp_sgd.shap_values(X_post_scaled)

shap_stability = {
    "Logistic Regression": shap_spearman(sv_sgd_pre,  sv_sgd_post),
    "Random Forest":       shap_spearman(sv_rf_pre,   sv_rf_post),
    "XGBoost":             shap_spearman(shap_pre,    shap_post),
}

# Summary table
rows = []
for name in auc_results:
    stab = shap_stability[name]
    if hasattr(stab, "__len__"):
        stab = np.asarray(stab).squeeze()
        if stab.size > 1:
            stab = stab.flat[0]
    rows.append({
        'Model':          name,
        'AUC Pre-COVID':  round(float(auc_results[name]['auc_pre']), 4),
        'AUC Post-COVID': round(float(auc_results[name]['auc_post']), 4),
        'AUC Drop':       round(float(auc_results[name]['auc_drop']), 4),
        'SHAP Stability': round(float(stab), 4),
    })
df_tradeoff = pd.DataFrame(rows)
print("\nInterpretability-Performance Tradeoff:")
print(df_tradeoff.to_string(index=False))
df_tradeoff.to_csv(os.path.join(plots_path, "tradeoff_results.csv"), index=False)

# Scatter plots
fig, ax = plt.subplots(figsize=(7, 5))
colors = ['steelblue', 'coral', 'seagreen']
for i, row in df_tradeoff.iterrows():
    ax.scatter(row['AUC Drop'], row['SHAP Stability'], s=160, color=colors[i], zorder=5)
    ax.annotate(row['Model'], (row['AUC Drop'], row['SHAP Stability']),
                textcoords="offset points", xytext=(8, 4), fontsize=10)
ax.set_xlabel('AUC-ROC Drop (Pre to Post-COVID)', fontsize=11)
ax.set_ylabel('SHAP Stability (Spearman rho)', fontsize=11)
ax.set_title('Interpretability-Performance Tradeoff Under Temporal Shift', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(plots_path, "tradeoff_plot.png"), dpi=150, bbox_inches='tight')
plt.close()
print("Saved tradeoff_plot.png and tradeoff_results.csv")

Logistic Regression        pre: 0.7664  post: 0.7086  drop: 0.0577
Random Forest              pre: 0.8414  post: 0.7486  drop: 0.0928
XGBoost                    pre: 0.8745  post: 0.7447  drop: 0.1298

Computing RF SHAP values...


[Parallel(n_jobs=32)]: Using backend ThreadingBackend with 32 concurrent workers.
[Parallel(n_jobs=32)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=32)]: Using backend ThreadingBackend with 32 concurrent workers.
[Parallel(n_jobs=32)]: Done 100 out of 100 | elapsed:    0.0s finished


Computing SGD SHAP values...

Interpretability-Performance Tradeoff:
              Model  AUC Pre-COVID  AUC Post-COVID  AUC Drop  SHAP Stability
Logistic Regression         0.7664          0.7086    0.0577          0.9195
      Random Forest         0.8414          0.7486    0.0928          1.0000
            XGBoost         0.8745          0.7447    0.1298          0.9441
Saved tradeoff_plot.png and tradeoff_results.csv


# Results Summary

In [13]:
import pandas as pd
import numpy as np
import os
from scipy.stats import spearmanr

models_path = os.path.join(data_path, "models")
plots_path  = os.path.join(data_path, "plots")

df_tradeoff = pd.read_csv(os.path.join(plots_path, "tradeoff_results.csv"))
shap_cw     = np.load(os.path.join(models_path, "shap_post.npy"))
shap_smote  = np.load(os.path.join(models_path, "shap_smote.npy"))
smote_corr, _ = spearmanr(
    np.abs(shap_cw).mean(axis=0),
    np.abs(shap_smote).mean(axis=0)
)


print("COMPLETE RESULTS SUMMARY")


print("\n1. MODEL PERFORMANCE")
performance = {
    'SGD Logistic Regression': {'AUC-ROC': 0.7189, 'Avg Precision': 0.2033},
    'Random Forest':            {'AUC-ROC': 0.7533, 'Avg Precision': 0.2304},
    'XGBoost (primary)':        {'AUC-ROC': 0.7504, 'Avg Precision': 0.2615},
}
for model, metrics in performance.items():
    print(f"  {model:30s}  AUC-ROC: {metrics['AUC-ROC']:.4f}  Avg Precision: {metrics['Avg Precision']:.4f}")

print("\n2. TEMPORAL DRIFT (Pre vs Post-COVID SHAP)")
print(f"  Spearman Rank Correlation: 0.9447  (p < 0.0001)")
print(f"  Interpretation: Strong structural preservation with localised drift")
print(f"  Biggest risers:  firstpmtdate +2, origdti +2, sellername +2")
print(f"  Biggest fallers: numborrowers -6, servicername -3, micancellationind -6")

print("\n3. BOOTSTRAP STABILITY (Within Pre-COVID)")
print(f"  Kendall's W: 0.9671  (30 bootstrap runs, n=5,000)")
print(f"  Verdict: STRONGLY STABLE  (threshold > 0.8)")
print(f"  Most stable:  firstpmtdate  (CV = 0.043)")
print(f"  Least stable: channel       (CV = 0.463)")

print("\n4. PERTURBATION ROBUSTNESS")
print(f"  {'Noise Level':15s} {'Spearman':12s} {'Top-5 Agreement':}")
print(f"  {'Baseline 0%':15s} {'1.0000':12s} 100%")
print(f"  {'5%':15s} {'0.9526':12s} 100%")
print(f"  {'10%':15s} {'0.9485':12s} 100%")
print(f"  {'15%':15s} {'0.9408':12s} 100%")
print(f"  Verdict: ROBUST to data quality noise")

print("\n5. SMOTE vs CLASS-WEIGHTING")
print(f"  SHAP Spearman correlation: {smote_corr:.4f}")
print(f"  Verdict: {'CONSISTENT' if smote_corr > 0.9 else 'DIVERGENT'} feature rankings across both strategies")

print("\n6. INTERPRETABILITY-PERFORMANCE TRADEOFF")
print(df_tradeoff[['Model','AUC Pre-COVID','AUC Post-COVID','AUC Drop','SHAP Stability']].to_string(index=False))

# save final summary csv
rows = []
for model, metrics in performance.items():
    rows.append([model, "AUC-ROC (test)",       metrics['AUC-ROC']])
    rows.append([model, "Avg Precision (test)", metrics['Avg Precision']])
rows.append(["XGBoost", "SHAP Temporal Drift (rho)",   0.9447])
rows.append(["XGBoost", "Bootstrap Stability (W)",     0.9671])
rows.append(["XGBoost", "Perturbation 5% (rho)",       0.9526])
rows.append(["XGBoost", "Perturbation 10% (rho)",      0.9485])
rows.append(["XGBoost", "Perturbation 15% (rho)",      0.9408])
rows.append(["XGBoost", "SMOTE vs CW SHAP (rho)",      round(smote_corr, 4)])

pd.DataFrame(rows, columns=["Model","Metric","Value"]).to_csv(
    os.path.join(plots_path, "final_results_summary.csv"), index=False)
print("\nSaved final_results_summary.csv")

COMPLETE RESULTS SUMMARY

1. MODEL PERFORMANCE
  SGD Logistic Regression         AUC-ROC: 0.7189  Avg Precision: 0.2033
  Random Forest                   AUC-ROC: 0.7533  Avg Precision: 0.2304
  XGBoost (primary)               AUC-ROC: 0.7504  Avg Precision: 0.2615

2. TEMPORAL DRIFT (Pre vs Post-COVID SHAP)
  Spearman Rank Correlation: 0.9447  (p < 0.0001)
  Interpretation: Strong structural preservation with localised drift
  Biggest risers:  firstpmtdate +2, origdti +2, sellername +2
  Biggest fallers: numborrowers -6, servicername -3, micancellationind -6

3. BOOTSTRAP STABILITY (Within Pre-COVID)
  Kendall's W: 0.9671  (30 bootstrap runs, n=5,000)
  Verdict: STRONGLY STABLE  (threshold > 0.8)
  Most stable:  firstpmtdate  (CV = 0.043)
  Least stable: channel       (CV = 0.463)

4. PERTURBATION ROBUSTNESS
  Noise Level     Spearman     Top-5 Agreement
  Baseline 0%     1.0000       100%
  5%              0.9526       100%
  10%             0.9485       100%
  15%             0.9408